<a href="https://colab.research.google.com/github/arthisri2810-dot/News-Popularity-Intelligence-System/blob/main/News_Popularity_Intelligence_System_using_Transformer_Based_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch nltk textstat scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re

from transformers import DistilBertTokenizer, DistilBertModel
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import textstat

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving News_dataset (1).csv to News_dataset (1).csv


In [ ]:
df = pd.read_csv('News_dataset (1).csv')

# normalize column names
df.columns = df.columns.str.strip().str.lower()

# now safe to use
df = df[['title', 'description']].dropna()
df['text'] = df['title'] + " " + df['description']

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [ ]:
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state[:,0,:].squeeze().numpy()

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

model = SentenceTransformer('all-MiniLM-L6-v2')

batch_size = 64
embeddings = []

texts = df['clean_text'].astype(str).apply(lambda x: x[:200]).tolist()

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]

    emb = model.encode(
        batch,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings.extend(emb)

df['embedding'] = embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100%|██████████| 3389/3389 [1:21:38<00:00,  1.45s/it]


In [ ]:
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:
import textstat

def lexical_diversity(text):
    words = text.split()
    return len(set(words)) / (len(words) + 1)

def urgency_score(text):
    keywords = ['breaking', 'urgent', 'alert', 'exclusive', 'now']
    return sum([1 for word in keywords if word in text])

def emotion_intensity(text):
    return abs(sia.polarity_scores(text)['compound'])

def readability(text):
    score = textstat.flesch_reading_ease(text)
    return score / 100   # 🔥 FIXED

def length_score(text):
    return len(text) / 500

In [ ]:
df['lexical'] = df['clean_text'].apply(lexical_diversity)
df['urgency'] = df['clean_text'].apply(urgency_score)
df['emotion'] = df['clean_text'].apply(emotion_intensity)
df['readability'] = df['clean_text'].apply(readability)
df['length'] = df['clean_text'].apply(length_score)

In [ ]:
df[['lexical','urgency','emotion','readability','length']].head()

,lexical,urgency,emotion,readability,length
0,0.600000,1,0.4939,0.227862,0.350
1,0.750000,0,0.0000,0.416474,0.354
2,0.714286,0,0.0000,0.227633,0.356
3,0.827586,0,0.1027,0.061936,0.394
4,0.666667,0,0.4019,0.093614,0.440


In [ ]:
df['readability'] = df['clean_text'].apply(readability)

In [ ]:
df['popularity_score'] = (
    0.25 * df['emotion'] +
    0.20 * df['urgency'] +
    0.20 * df['lexical'] +
    0.15 * df['readability'] +
    0.20 * df['length']
)

In [ ]:
df[['popularity_score']].head()

,popularity_score
0,0.547654
1,0.283271
2,0.248202
3,0.279283
4,0.335850


In [ ]:
import numpy as np
import torch

X = np.vstack(df['embedding'].values)
y = df['popularity_score'].values

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1,1)

In [ ]:
import torch.nn as nn

class PopularityModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.fc(x)

model_nn = PopularityModel(384)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.001)

In [ ]:
for epoch in range(5):
    optimizer.zero_grad()

    outputs = model_nn(X_tensor)
    loss = criterion(outputs, y_tensor)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.14102467894554138
Epoch 2, Loss: 0.12984851002693176
Epoch 3, Loss: 0.11917093396186829
Epoch 4, Loss: 0.10894452780485153
Epoch 5, Loss: 0.09913215786218643


In [ ]:
df['predicted_score'] = model_nn(X_tensor).detach().numpy()

In [ ]:
df_sorted = df.sort_values(by='predicted_score', ascending=False)

df_sorted[['title','predicted_score']].head(10)

,title,predicted_score
710,NOW COMES FEDSPEAK: Here's a preview of this w...,0.214122
4474,GLOBAL ECONOMY WEEKAHEAD-Europe seen brushing ...,0.212445
2641,G20 turns to economy after terror talks,0.211236
61302,App developers support Obama Pacific Rim trade...,0.210706
58685,Profit From The No. 1 Threat To The Global Eco...,0.209551
177713,Labor chief asked: Why are OFWs still in Iraq?,0.208496
15627,Obama Admits He Failed to Respond to Terror At...,0.207672
54674,What Turnbull can learn from Obama's tech vision,0.207627
93578,"Hackers are getting smarter, says Ballmer",0.207013
19885,"On Economy, AP Wraps Up Year With Two Weak Sto...",0.206268


In [ ]:
def explain(index):
    row = df.iloc[index]

    print("📰 Title:", row['title'])
    print("🔥 Score:", row['predicted_score'])
    print("\n--- Why this score? ---")
    print("Emotion:", row['emotion'])
    print("Urgency:", row['urgency'])
    print("Lexical:", row['lexical'])
    print("Readability:", row['readability'])

In [ ]:
explain(0)

📰 Title: Obama Lays Wreath at Arlington National Cemetery
🔥 Score: 0.11153962

--- Why this score? ---
Emotion: 0.4939
Urgency: 1
Lexical: 0.6
Readability: 0.22786206896551733


In [ ]:
torch.save(model_nn.state_dict(), "model.pth")

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.0 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
from sentence_transformers import SentenceTransformer
import textstat
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

# ---------------- SETUP ---------------- #
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

st.set_page_config(
    page_title="News Popularity Intelligence System",
    page_icon="📰",
    layout="centered"
)

# Header
st.markdown(
    "<h1 style='text-align: center;'>📰 News Popularity Intelligence System</h1>",
    unsafe_allow_html=True
)
st.markdown("---")

# ---------------- MODEL ---------------- #
class PopularityModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.fc(x)

model_nn = PopularityModel(384)
model_nn.load_state_dict(torch.load("model.pth", map_location=torch.device('cpu')))
model_nn.eval()

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# ---------------- FUNCTIONS ---------------- #
def clean_text(text):
    return text.lower()

def lexical_diversity(text):
    words = text.split()
    return len(set(words)) / (len(words) + 1)

# 🔥 FIXED URGENCY
def urgency_score(text):
    keywords = ['breaking', 'urgent', 'alert', 'exclusive', 'now']
    return 1 if any(word in text for word in keywords) else 0

def emotion_intensity(text):
    return abs(sia.polarity_scores(text)['compound'])

def readability(text):
    return textstat.flesch_reading_ease(text) / 100

def length_score(text):
    return len(text) / 500

# 🔥 FINAL SMART LABEL FUNCTION
def get_label(score, explanation):
    if explanation["Urgency"] == 1 and explanation["Emotion"] > 0.55:
        return "🔥 Breaking / Highly Popular"
    elif score >= 0.60:
        return "🔥 Breaking / Highly Popular"
    elif score >= 0.40:
        return "⚡ Trending / Medium Popular"
    else:
        return "🧊 Low Popularity"

# 🔥 FINAL HYBRID PREDICTION
def predict(title, desc):
    text = clean_text(title + " " + desc)

    # Embedding
    emb = embedder.encode(text, convert_to_numpy=True)
    emb_tensor = torch.tensor(emb, dtype=torch.float32)

    # Model score
    model_score = model_nn(emb_tensor).item()

    # 🔥 IMPROVED PROXY SCORE
    proxy_score = (
        0.35 * emotion_intensity(text) +
        0.25 * urgency_score(text) +
        0.15 * lexical_diversity(text) +
        0.10 * readability(text) +
        0.15 * length_score(text)
    )

    # 🔥 FINAL SCORE
    score = 0.2 * model_score + 0.8 * proxy_score

    explanation = {
        "Emotion": emotion_intensity(text),
        "Urgency": urgency_score(text),
        "Lexical": lexical_diversity(text),
        "Readability": readability(text),
        "Length": length_score(text)
    }

    return score, explanation

# ---------------- SIDEBAR ---------------- #
st.sidebar.title("📌 Navigation")
page = st.sidebar.radio("Go to", ["Home", "News Intelligence", "Model Reasoning"])

# ---------------- HOME ---------------- #
if page == "Home":
    st.write("""
    ### 📌 Problem Overview

    This system predicts **news popularity** using Transformer-based deep learning
    without labeled data.

    ### 🔍 Key Idea
    Popularity is a **latent variable** inferred from:
    - Emotional intensity
    - Urgency
    - Linguistic richness
    - Readability

    ### ⚙️ Architecture
    Transformer → Embeddings → Proxy Signals → Hybrid Scoring Model
    """)

# ---------------- NEWS INTELLIGENCE ---------------- #
elif page == "News Intelligence":
    st.subheader("🔍 Analyze News Popularity")

    title = st.text_input("Enter News Title")
    desc = st.text_area("Enter Description")

    if st.button("Analyze"):
        if title.strip() == "" or desc.strip() == "":
            st.warning("Please enter both title and description")
        else:
            score, explanation = predict(title, desc)

            # 🔥 CORRECT FUNCTION CALL
            label = get_label(score, explanation)

            # Output
            st.success(f"🔥 Popularity Score: {score:.2f}")
            st.markdown(f"### 🏷️ {label}")

            # Progress bar
            st.progress(float(min(score, 1.0)))

            # Signals
            st.write("### 🧠 Key Signals")
            for k, v in explanation.items():
                st.write(f"**{k}**: {v:.3f}")

# ---------------- MODEL REASONING ---------------- #
elif page == "Model Reasoning":
    st.subheader("🧠 Model Reasoning")

    st.write("""
    ### 🤖 How the Model Works

    This system uses **self-supervised learning**.

    Since no labels exist, we generate proxy signals:

    - Emotion → engagement
    - Urgency → breaking news detection
    - Lexical diversity → richness
    - Readability → accessibility
    - Length → depth

    ### 🔥 Hybrid Intelligence
    Final score combines:
    - Transformer model output
    - Proxy-based scoring
    - Rule-based reasoning

    This improves reliability and interpretability.
    """)

Writing app.py


In [ ]:
!pip install pyngrok

In [ ]:
!streamlit run app.py &>/dev/null&

In [ ]:
!ngrok authtoken 36IeTvr7PlB5X7b8Kv3Kfw6FF9r_7DoFqQJNdrUww2ADZ5jFq

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("Click this link:", public_url)

Click this link: NgrokTunnel: "https://unreducible-twanda-unconsumptive.ngrok-free.dev" -> "http://localhost:8501"
